# A1_build_retrieval_index

In [1]:
import os; os.getcwd()

'/Users/mandy/Documents/ColumbiaU/5560 applied generatvie ai/project'

In [2]:
# step 1: Build an index on sample data.
# Step 1: Load the cleaned Bright Data reviews and inspect their structure.
import pandas as pd
df = pd.read_csv("Data_cleaning_export/data/processed/brightdata_recent_clean_Reviews.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nDataset Head:")
display(df.head())

Dataset shape: (431, 40)

Column names:
['timestamp', 'overview_id', 'review_id', 'review_url', 'date_review', 'count_helpful', 'count_unhelpful', 'employee_job_end_year', 'employee_length', 'employee_responses', 'employee_status', 'current', 'flag_covid', 'flag_featured', 'outlook', 'ceo_approv', 'recommend', 'culture_values', 'diversity_inclusion', 'overall_rating', 'work_life_balance', 'headline', 'firm', 'review_advice', 'url', 'location', 'job_title', 'advice_to_management', 'overview_url', 'original_url', 'pros', 'cons', 'comp_benefits', 'senior_mgmt', 'career_opp', 'glassdoor_employer_id', 'source', 'review_text', 'text_length', 'sentiment_label']

Dataset Head:


,timestamp,overview_id,review_id,review_url,date_review,count_helpful,count_unhelpful,employee_job_end_year,employee_length,employee_responses,...,pros,cons,comp_benefits,senior_mgmt,career_opp,glassdoor_employer_id,source,review_text,text_length,sentiment_label
0,2024-04-02 00:00:00.000,100231,59978917,NaN,2022-02-19 21:38:00.363000+00:00,2.0,NaN,NaN,6.0,NaN,...,1. Great brand name 2. Top Management are cons...,1. Work hours do not adhere to internationally...,3,5,5.0,100231.0,brightdata_glassdoor,TITLE: Great Company to work with PROS: 1. Gre...,378,positive
1,2023-07-28 00:00:00.000,58316203,58316203,NaN,2022-01-21 00:00:00+00:00,1.0,NaN,2021.0,2.0,NaN,...,Very inclusive and helpful environment which m...,Nothing sticks out to me that I would change.,5,5,NaN,NaN,brightdata_glassdoor,TITLE: Commercial Product Internship PROS: Ver...,184,positive
2,2023-08-03 00:00:00.000,42257812,77730254,NaN,2023-06-26 00:00:00+00:00,3.0,NaN,2023.0,2.0,NaN,...,- Eye-opening: Teaches you exactly how a compa...,- Communication breakdown: There is an endemic...,2,1,NaN,NaN,brightdata_glassdoor,TITLE: Regrettable Experience - A Workplace To...,754,negative
3,2023-07-23 00:00:00.000,70172378,70172378,NaN,2022-10-15 00:00:00+00:00,1.0,NaN,2022.0,4.0,NaN,...,They are okay to work for.,Don't expect to be treated fairly from Distric...,5,2,NaN,NaN,brightdata_glassdoor,TITLE: Don't expect to advance at your home st...,157,neutral
4,2023-07-31 00:00:00.000,60910653,60910653,NaN,2022-03-10 00:00:00+00:00,4.0,NaN,2021.0,1.0,NaN,...,provides good salary package on time,too costlier products bad and each 6 months ch...,1,1,NaN,NaN,brightdata_glassdoor,TITLE: Worse Company Ever PROS: provides good ...,525,negative


In [3]:
# step 2: check data quality
# step 2.1 dates
df["date"] = pd.to_datetime(df["date_review"], format="ISO8601", utc=True)
df["year"] = df["date"].dt.year

print("Failed to parse:", df["date"].isna().sum())
print(df["year"].value_counts().sort_index())

Failed to parse: 0
year
2022    357
2023     66
2024      8
Name: count, dtype: int64


#### 2024 has only 8 reviews, too few to support conclusions about that year. When asked about 2024, the system should say the evidence is limited instead of producing a confident summary. This becomes a test case for the Q&A set (insufficient evidence).


In [4]:
# step 2.1 Check review IDs
print("Missing IDs:", df["review_id"].isna().sum())
print("All unique:", df["review_id"].is_unique)
print("ID type:", df["review_id"].dtype)
print(df["review_id"].head())

Missing IDs: 0
All unique: True
ID type: int64
0    59978917
1    58316203
2    77730254
3    70172378
4    60910653
Name: review_id, dtype: int64


#### review IDs are unique and have no missing values. it will be the basis for passage IDs and citations, and for joining classification results.

In [5]:
# step 2.2: Check text fields
text_cols = ["pros", "cons", "advice_to_management"]

for col in text_cols:
    words = df[col].fillna("").str.split().str.len()
    print(f"--- {col} ---")
    print("Missing:", df[col].isna().sum())
    print("Fewer than 3 words:", (words < 3).sum())
    print("Over 150 words:", (words > 150).sum())
    print(words.describe().round(1))
    print()

--- pros ---
Missing: 0
Fewer than 3 words: 0
Over 150 words: 2
count    431.0
mean      17.3
std       23.7
min        5.0
25%        6.0
50%        9.0
75%       18.0
max      183.0
Name: pros, dtype: float64

--- cons ---
Missing: 0
Fewer than 3 words: 0
Over 150 words: 54
count    431.0
mean      63.8
std       96.3
min        5.0
25%       10.0
50%       26.0
75%       72.5
max      646.0
Name: cons, dtype: float64

--- advice_to_management ---
Missing: 305
Fewer than 3 words: 327
Over 150 words: 1
count    431.0
mean       5.1
std       19.8
min        0.0
25%        0.0
50%        0.0
75%        2.0
max      335.0
Name: advice_to_management, dtype: float64



#### Passage rules: skip text under 3 words; split text over 150 words into chunks.

In [6]:
# step 2.3: Normalize employment status
status = df["current"].fillna("").str.lower()

df["employment_status"] = "unknown"
df.loc[status.str.startswith("current"), "employment_status"] = "current"
df.loc[status.str.startswith("former"), "employment_status"] = "former"

print(df["employment_status"].value_counts())

employment_status
former     409
current     17
unknown      5
Name: count, dtype: int64


#### Bright Data is almost entirely former employees (~409 of 431; ~17 current). A current-vs-former comparison is not meaningful on this corpus and is left to the Kaggle demo corpus.

In [7]:
# step 3.1: Split each review into passages (pros / cons / advice)
fields = {"pros": "pros", "cons": "cons", "advice": "advice_to_management"}
meta_cols = ["firm", "year", "employment_status", "job_title", "overall_rating", "sentiment_label"]

rows = []
for _, review in df.iterrows():
    for field, col in fields.items():
        text = review[col]
        if pd.isna(text) or len(str(text).split()) < 3:
            continue
        row = {
            "passage_id": f"brightdata:{review['review_id']}:{field}",
            "review_id": review["review_id"],
            "field": field,
            "text": str(text).strip(),
        }
        for c in meta_cols:
            row[c] = review[c]
        rows.append(row)

passages = pd.DataFrame(rows)
print(passages.shape)
print(passages["field"].value_counts())
print("IDs unique:", passages["passage_id"].is_unique)
passages.head()

(966, 10)
field
pros      431
cons      431
advice    104
Name: count, dtype: int64
IDs unique: True


,passage_id,review_id,field,text,firm,year,employment_status,job_title,overall_rating,sentiment_label
0,brightdata:59978917:pros,59978917,pros,1. Great brand name 2. Top Management are cons...,HDFC Bank,2022,current,Assistant Vice President,5,positive
1,brightdata:59978917:cons,59978917,cons,1. Work hours do not adhere to internationally...,HDFC Bank,2022,current,Assistant Vice President,5,positive
2,brightdata:59978917:advice,59978917,advice,Bring this great Company at par with similar g...,HDFC Bank,2022,current,Assistant Vice President,5,positive
3,brightdata:58316203:pros,58316203,pros,Very inclusive and helpful environment which m...,Florida Blue,2022,former,NaN,5,positive
4,brightdata:58316203:cons,58316203,cons,Nothing sticks out to me that I would change.,Florida Blue,2022,former,NaN,5,positive


#### notes:
- Unit of retrieval: one passage = one field (pros, cons or advice) of one review.
- Keeping pros and cons separate matters: a single review often praises pay in "pros" and criticizes management in "cons". Answers must cite the right side.
- Passage ID format: `source:review_id:field` (e.g., `brightdata:59978917:cons`). The ID shows the dataset, the review and the field at a glance, and stays unique when Kaggle passages are added.

In [8]:
# step3.2： Split long passages into chunks of up to 150 words
def split_text(text, max_words=150, overlap=30):
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks = []
    step = max_words - overlap
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + max_words]))
        if start + max_words >= len(words):
            break
    return chunks

passages["text"] = passages["text"].apply(split_text)
passages = passages.explode("text").reset_index(drop=True)

passages["chunk"] = passages.groupby("passage_id").cumcount()
passages["passage_id"] = passages["passage_id"] + ":" + passages["chunk"].astype(str)

print(passages.shape)
print(passages["field"].value_counts())
print("IDs unique:", passages["passage_id"].is_unique)
print("Longest chunk (words):", passages["text"].str.split().str.len().max())
print("Passages split into chunks:", (passages["chunk"] > 0).sum())

(1058, 11)
field
cons      519
pros      433
advice    106
Name: count, dtype: int64
IDs unique: True
Longest chunk (words): 150
Passages split into chunks: 92


#### chunking split into windows of up to 150 words with a 30-word overlap, so a sentence cut at a boundary still appears whole in one chunk.
- Passage ID format becomes `source:review_id:field:chunk` (e.g., `brightdata:59978917:cons:1`).

In [10]:
# Step 4: Turn each passage into a vector
from sentence_transformers import SentenceTransformer
import numpy as np
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
embeddings = model.encode(
    passages["text"].tolist(),
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Shape:", embeddings.shape)
print("Type:", embeddings.dtype)
print("Length of first vector:", np.linalg.norm(embeddings[0]).round(3))

W1114 21:20:44.388000 23667 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Shape: (1058, 384)
Type: float32
Length of first vector: 1.0


#### Normalized vectors above

In [11]:
# Step 5: Save the index to files
import json
from pathlib import Path

index_dir = Path("indexes/brightdata_eval")
index_dir.mkdir(parents=True, exist_ok=True)

passages.to_parquet(index_dir / "passages.parquet", index=False)
np.save(index_dir / "embeddings.npy", embeddings)

info = {
    "corpus": "brightdata_eval",
    "source_file": "brightdata_recent_clean_Reviews.csv",
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "n_reviews": int(passages["review_id"].nunique()),
    "n_passages": len(passages),
    "dims": embeddings.shape[1],
    "max_words": 150,
    "overlap_words": 30,
}
with open(index_dir / "manifest.json", "w") as f:
    json.dump(info, f, indent=2)

print(sorted(p.name for p in index_dir.iterdir()))

['embeddings.npy', 'manifest.json', 'passages.parquet']


In [12]:
# Step 6: Load the index and run a test search
index_dir = Path("indexes/brightdata_eval")

passages = pd.read_parquet(index_dir / "passages.parquet")
embeddings = np.load(index_dir / "embeddings.npy")
with open(index_dir / "manifest.json") as f:
    info = json.load(f)

model = SentenceTransformer(info["model"], device="cpu")
print("Loaded:", passages.shape, embeddings.shape)


def search(query, k=5, field=None):
    q = model.encode(query, normalize_embeddings=True)
    scores = embeddings @ q

    results = passages.copy()
    results["score"] = scores
    if field:
        results = results[results["field"] == field]
    return results.sort_values("score", ascending=False).head(k)


cols = ["passage_id", "score", "firm", "year", "text"]
search("unfair promotions and no career growth", k=5, field="cons")[cols]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded: (1058, 11) (1058, 384)


,passage_id,score,firm,year,text
200,brightdata:75112452:cons:0,0.688885,Teleperformance,2023,Promotion process is not fair
616,brightdata:65867577:cons:0,0.637518,Scotiabank,2022,1. Salaries too low. 2. Hard to get a promotio...
881,brightdata:82772565:cons:0,0.596237,NAB - National Australia Bank,2023,No Career Growth No Salary Hike Minimal Bonus
244,brightdata:77695149:cons:0,0.593519,JPMorgan Chase & Co,2023,* Very slow growth in Salary and less promotio...
473,brightdata:65675985:cons:0,0.592363,Amazon,2022,Too Much Favoritism for promotions Area Manage...


#### All top 5 results are relevant.
#### Possible length bias: all top results are short. Short passages focus on one topic and score higher; long multi-topic chunks may be pushed down. To check in A2 (BM25 vs embeddings).
#### Some passages mix topics (salary and promotion); the generator should cite only the relevant part.

In [13]:
search("good benefits and pay", k=5, field="pros")[cols]

,passage_id,score,firm,year,text
1013,brightdata:64038477:pros:0,0.874547,Fisher Investments,2022,"Good Benefits, some good people to work with"
42,brightdata:79596882:pros:0,0.821049,JPMorgan Chase & Co,2023,"Really good benefits including healthcare, vac..."
913,brightdata:65381371:pros:0,0.800573,UPS,2022,Good health benefit and good people to work with.
191,brightdata:60327976:pros:0,0.782019,Splunk,2022,Good pay and benefits. Coworkers are helpful.
994,brightdata:64031984:pros:0,0.763869,DeGolyer and MacNaughton,2022,"Good pay, health insurance and retirement plan."


In [14]:
search("long hours and burnout", k=5, field="cons")[cols]

,passage_id,score,firm,year,text
238,brightdata:82771804:cons:0,0.617195,JPMorgan Chase & Co,2023,long working hours sometime at office
1010,brightdata:62686728:cons:0,0.585426,Jushi,2022,"overworked, burnout is rampant, management is ..."
997,brightdata:60285286:cons:0,0.574215,Warburg Pincus,2022,very long hours and little focus on fixing it ...
766,brightdata:59264505:cons:0,0.500130,City Furniture,2022,Too much work no time for family
957,brightdata:65487553:cons:0,0.496912,Covanta,2022,Long hours Hard manual labor Frequent superflu...


#### Search check
- Relevant top 5 for promotion, overtime and benefits queries; embeddings match meaning ("favoritism" ≈ unfair promotions).
- Scores are not comparable across queries (~0.8 for benefits vs ~0.5–0.7 for complaints), so a fixed threshold cannot decide whether evidence exists.


### A2: BM25 vs embedding search
Goal: compare the two retrieval methods and decide which to use (or combine).

In [17]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [23]:
# step 2.1 retrieval module： create .py and import to notebook
%load_ext autoreload
%autoreload 2

from retrieval.search import load_index, semantic_search, bm25_search

index = load_index("indexes/brightdata_eval")
cols = ["passage_id", "score", "firm", "text"]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [28]:
# A2.2 simple Example: "unfair promotions and no career growth" (cons)"
- Semantic: 5/5 relevant. BM25: ~2/5; top hit only matched "unfair".
- BM25 missed "Promotion process is not fair" (no exact word match).
query = "unfair promotions and no career growth"

print("=== BM25 ===")
display(bm25_search(index, query, k=5, field="cons")[cols])

print("=== Semantic ===")
display(semantic_search(index, query, k=5, field="cons")[cols])

=== BM25 ===


,passage_id,score,firm,text
550,brightdata:66495058:cons:0,9.849102,Oaks Integrated Care,"Horrible management, low pay, unfair"
881,brightdata:82772565:cons:0,9.474263,NAB - National Australia Bank,No Career Growth No Salary Hike Minimal Bonus
296,brightdata:68403371:cons:1,7.089840,C2 Education,:O -High turnover rate of education coordinato...
340,brightdata:59116734:cons:0,5.873779,Praytell Strategy,• there’s so clearly one side of the business ...
854,brightdata:71785647:cons:0,5.353334,Reliance Industries,Everything is a con. You think of any aspect o...


=== Semantic ===


,passage_id,score,firm,text
200,brightdata:75112452:cons:0,0.688885,Teleperformance,Promotion process is not fair
616,brightdata:65867577:cons:0,0.637518,Scotiabank,1. Salaries too low. 2. Hard to get a promotio...
881,brightdata:82772565:cons:0,0.596237,NAB - National Australia Bank,No Career Growth No Salary Hike Minimal Bonus
244,brightdata:77695149:cons:0,0.593519,JPMorgan Chase & Co,* Very slow growth in Salary and less promotio...
473,brightdata:65675985:cons:0,0.592363,Amazon,Too Much Favoritism for promotions Area Manage...


#### manual comparison result:
Semantic: 5/5 relevant. BM25: ~2/5; top hit only matched "unfair"; missed "Promotion process is not fair" (no exact word match,promotion ≠ promotions, fair ≠ unfair).
- Opposite length bias: BM25 favors long passages (more word repeats); embeddings favor short, focused ones. This suggests a hybrid may help.
- Fairness: BM25 without stemming is a weak baseline. A stemmed BM25 will be added so the comparison is not one-sided.

BM25
- Standard keyword (sparse) search: rewards query words that appear often in a passage and are rare overall; discounts long passages.
- Tokenize: lowercase, letters/digits only, remove stop words.
- Matches identical words only ("promotions" ≠ "promotion", "unfair" ≠ "fair"). Scores are not on a 0–1 scale; compare rankings, not scores.

In [30]:
# A2.3 complex Example
tests = [
    # (type, query, field, filters)
    ("exact term",    "401k match",                              None,   None),
    ("exact term",    "micromanagement",                         "cons", None),
    ("acronym",       "not enough PTO",                          "cons", None),
    ("acronym",       "problems with HR",                        "cons", None),
    ("paraphrase",    "my boss watches everything I do",         "cons", None),
    ("paraphrase",    "employees are underpaid",                 "cons", None),
    ("multi-topic",   "long hours and low pay",                  "cons", None),
    ("direction",     "complaints about salary",                 "cons", None),
    ("direction",     "what employees like about salary",        "pros", None),
    ("metadata",      "bad management",                          "cons", {"year": 2023}),
    ("unanswerable",  "no childcare or daycare support",         "cons", None),
    ("unanswerable",  "visa sponsorship for international staff", None,  None),
]

for qtype, query, field, filters in tests:
    print(f"\n##### [{qtype}] {query} | field={field} filters={filters}")
    for name, search_fn in [("BM25", bm25_search), ("Semantic", semantic_search)]:
        print(f"--- {name}")
        hits = search_fn(index, query, k=3, field=field, filters=filters)
        for _, h in hits.iterrows():
            print(f"   - [{h['field']}, {h['year']}] {h['text'][:110]}")


##### [exact term] 401k match | field=None filters=None
--- BM25
   - [pros, 2022] SHC has good 401k match
   - [pros, 2022] Knowledgeable employees, great 401k match, great benefits
   - [pros, 2022] The Fed has so many programs available to employees for professional development. On the IT side, there are pr
--- Semantic
   - [pros, 2022] SHC has good 401k match
   - [pros, 2022] Knowledgeable employees, great 401k match, great benefits
   - [cons, 2023] Payment below to the competitors in the same positions

##### [exact term] micromanagement | field=cons filters=None
--- BM25
   - [cons, 2022] Supervisors, broke system, micromanagement, computer issues, lack of support, everything
   - [cons, 2022] Hard to focus on your work while dealing with the reporting and training demands. Some managers are into old s
   - [cons, 2022] everything, Micromanagement, bad WLB, no shift allowance, No leave encashment, no smooth existing process, Var
--- Semantic
   - [cons, 2022] insane micromana

#### Manual review: 12 queries, 7 types (top 3, relevant = topic and direction match)

| Type | Query | BM25 | Semantic |
|---|---|---|---|
| exact term | 401k match | 2–3 | 2 |
| exact term | micromanagement (cons) | ~3 | 2–3 |
| acronym | not enough PTO | 2–3 | 0 |
| acronym | problems with HR | 1–2 | 3 |
| paraphrase | my boss watches everything I do | 0 | 0 |
| paraphrase | employees are underpaid | 3 | 3 |
| multi-topic | long hours and low pay | 2–3 | 2–3 |
| direction | complaints about salary (cons) | 1 | 3 |
| direction | what employees like about salary (pros) | 2 | 3 |
| metadata | bad management (cons, 2023) | 1 | 2 |
| unanswerable | childcare / visa sponsorship | – | – |
| **Total (10 answerable, max 30)** | | **~18** | **~21** |

- Semantic is slightly ahead overall; BM25 wins on PTO, semantic on HR and direction questions.
- BM25 treats every word in the question as content. Words that only describe the request ("complaints", "like", "problems") pull in wrong passages: for "complaints about salary", its top hit was "No complaints as of yet".
- Both methods fail on a hard paraphrase: "my boss watches everything I do" describes micromanagement, but shares no words with the 13 reviews that mention it, and neither method found them.
- Both always return k results, even when no answer exists; refusal must be handled after retrieval.
- Metadata filters work (all results from 2023).

In [29]:
# A2.4 further look into PTO
q = "not enough PTO"
print("BM25 (cons):", bm25_search(index, q, k=3, field="cons")["text"].str[:120].tolist())
print("Semantic (cons):", semantic_search(index, q, k=3, field="cons")["text"].str[:120].tolist())
print("Semantic, reworded:", semantic_search(index, "not enough vacation or paid time off", k=3, field="cons")["text"].str[:120].tolist())

BM25 (cons): ['I left a review of how I ask off it was for my daughters wedding. Then it was answered I could have used my PTO sick tim', "I was kept as a temp for over a year, continually promised to get hired on as permanent but it didn't happen for over a ", 'There are not really advancement opportunities within the team outside of tech lead, which is held over us to keep us on']
Semantic (cons): ['A recent Bloomberg article about the CTO is a reflection of the technology culture and how highly biased the technology ', 'too much admin and paper work', 'Demands way too much in multiple ways.']
Semantic, reworded: ['Too much work no time for family', "Self checkout work wouldn't give time off", 'Low pay, overworked, no support']


#### Field filter is necessary but not sufficient
- With `field="cons"`, "not enough PTO" no longer returns praise, but the embedding results are unrelated ("too much admin and paper work", "Demands way too much").
- Relevant complaints exist: 17 cons passages mention PTO, vacation or time off, but mostly as one sentence inside long, multi-topic complaints.
- Why embeddings miss them: (1) topic dilution — one vector for a 150-word, multi-topic chunk underweights a single sentence about PTO, while short generic complaints that sound like "not enough" score higher; (2) the small model may not represent the acronym "PTO" well.
- BM25 is strong exactly here: it finds the exact term regardless of passage length.
- The two methods fail in complementary ways, which motivates testing a hybrid (BM25 + embeddings) as a third method in A2.

In [31]:
# A2.5 further check scores on answerable vs unanswerable
import pandas as pd

rows = []
for qtype, query, field, filters in tests:
    row = {"type": qtype, "query": query}
    for name, search_fn in [("bm25", bm25_search), ("semantic", semantic_search)]:
        hits = search_fn(index, query, k=1, field=field, filters=filters)
        row[f"{name}_top1"] = round(hits["score"].iloc[0], 3)
    rows.append(row)

score_table = pd.DataFrame(rows)
score_table

,type,query,bm25_top1,semantic_top1
0,exact term,401k match,17.898,0.677
1,exact term,micromanagement,6.763,0.693
2,acronym,not enough PTO,2.315,0.364
3,acronym,problems with HR,6.080,0.612
4,paraphrase,my boss watches everything I do,7.775,0.468
5,paraphrase,employees are underpaid,6.591,0.666
6,multi-topic,long hours and low pay,14.534,0.681
7,direction,complaints about salary,8.208,0.530
8,direction,what employees like about salary,5.774,0.597
9,metadata,bad management,7.214,0.586


#### Top-1 score check (answerable vs unanswerable)
| Type | Query | BM25 top-1 | Semantic top-1 |
|---|---|---|---|
| exact term | 401k match | 17.898 | 0.677 |
| exact term | micromanagement | 6.763 | 0.693 |
| acronym | not enough PTO | 2.315 | 0.364 |
| acronym | problems with HR | 6.080 | 0.612 |
| paraphrase | my boss watches everything I do | 7.775 | 0.468 |
| paraphrase | employees are underpaid | 6.591 | 0.666 |
| multi-topic | long hours and low pay | 14.534 | 0.681 |
| direction | complaints about salary | 8.208 | 0.530 |
| direction | what employees like about salary | 5.774 | 0.597 |
| metadata | bad management (2023) | 7.214 | 0.586 |
| unanswerable | no childcare or daycare support | 5.422 | 0.424 |
| unanswerable | visa sponsorship for international staff | 8.490 | 0.408 |

- BM25 scores cannot detect missing answers: unanswerable queries fall mid-range, and the lowest score (PTO) was a query BM25 answered correctly.
- Semantic top-1 below ~0.5 matched exactly the 4 queries where semantic found nothing relevant (2 unanswerable, PTO, boss paraphrase).
- A low semantic score means "semantic found nothing", not "no answer exists" (BM25 found PTO). Use it as a low-confidence signal (e.g., fall back to BM25), not as the refusal rule.
- Only 12 queries; the dev set needs 5–8 unanswerable questions to test the ~0.5 cutoff.

### A4. Evaluation

In [32]:
from retrieval.evaluate import recall_at_k, reciprocal_rank, evaluate

retrieved = ["p3", "p7", "p1", "p9", "p2"]   # search results, best first
gold = ["p1", "p2", "p8"]                     # passages marked relevant by a person

print(recall_at_k(retrieved, gold, 3))   # expect 0.333
print(recall_at_k(retrieved, gold, 5))   # expect 0.667
print(reciprocal_rank(retrieved, gold))  # expect 0.333

0.3333333333333333
0.6666666666666666
0.3333333333333333


In [33]:
toy = pd.DataFrame([
    {"question_id": "t1", "query_type": "paraphrase",
     "question": "unfair promotions and no career growth", "field": "cons", "year": None,
     "gold_ids": "brightdata:75112452:cons:0;brightdata:65675985:cons:0"},
    {"question_id": "t2", "query_type": "unanswerable",
     "question": "visa sponsorship for international staff", "field": None, "year": None,
     "gold_ids": None},
])

print("=== Semantic ===")
display(evaluate(semantic_search, index, toy))
print("=== BM25 ===")
display(evaluate(bm25_search, index, toy))

=== Semantic ===


,question_id,query_type,top1_score,recall@5,recall@10,rr
0,t1,paraphrase,0.688885,1.0,1.0,1.0
1,t2,unanswerable,0.408168,NaN,NaN,NaN


=== BM25 ===


,question_id,query_type,top1_score,recall@5,recall@10,rr
0,t1,paraphrase,9.849102,0.0,0.0,0.0
1,t2,unanswerable,8.490423,NaN,NaN,NaN


#### 
- `recall_at_k`: share of gold passages in the top k. `reciprocal_rank`: 1 / rank of the first gold passage.
- `evaluate(search_fn, index, questions)`: scores any search method on a question table (columns: question_id, query_type, question, field, year, gold_ids).
- Unanswerable questions (no gold) skip recall/MRR; only the top-1 score is kept.